In [ ]:
%pip install -U openai pydantic pydantic-core tqdm


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, time, datetime
import pandas as pd
import numpy as np
import requests
from getpass import getpass
from tqdm.auto import tqdm

MODEL = "gpt-5.4-nano"    
PROMPT = open("ai_labels/wordcheck_prompt.txt", encoding="utf-8").read()  
LIST_ORDER = ["anger", "sadness", "negative", "positive", "money"]
API_URL = "https://api.openai.com/v1/chat/completions"  

print("requests version:", requests.__version__)
print("Prompt loaded:", len(PROMPT.split()), "words")

requests version: 2.32.3
Prompt loaded: 185 words


In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")   
HEADERS = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
           "Content-Type": "application/json"}
SETTINGS = {"temperature": True}  

def ai_label(lst, snippet):
    """Send one snippet to the model. Returns ('Y' or 'N', model name actually used)."""
    for attempt in range(4):
        body = {"model": MODEL,
                "messages": [{"role": "system", "content": PROMPT},
                             {"role": "user", "content": f"List: {lst}\nSnippet: {snippet}"}]}
        if SETTINGS["temperature"]:
            body["temperature"] = 0
        try:
            r = requests.post(API_URL, headers=HEADERS, json=body, timeout=60)
        except requests.RequestException as e:
            print("Network problem:", e, "-> retrying in 5 seconds")
            time.sleep(5)
            continue
        if r.status_code == 200:                      
            data = r.json()
            return data["choices"][0]["message"]["content"].strip().upper()[:1], data["model"]
        error = r.text
        if SETTINGS["temperature"] and "temperature" in error.lower():
            print("Model does not accept temperature=0 -> using its default.")
            SETTINGS["temperature"] = False
            continue
        print("API error", r.status_code, ":", error[:300])
        if r.status_code in (401, 403, 404):         
            break
        time.sleep(5)
    return "", ""

In [4]:
wc_test = pd.read_csv("ai_labels/word_check.csv", encoding="utf-8-sig", keep_default_na=False)
tests = [("anger", "smell"), ("positive", "love"), ("money", "price"),
         ("sadness", "dark"), ("negative", "broken")]
for lst, word in tests:
    row = wc_test[(wc_test["list"] == lst) & (wc_test["word"] == word)].iloc[0]
    label, used = ai_label(lst, row["snippet"])
    print(f"{lst:8s} {word:8s} -> {label}   | {row['snippet'][:70]}")
print("\nModel actually used:", used, "| temperature=0:", SETTINGS["temperature"])

anger    smell    -> N   | ...ent brands to use in my cold process soap. this did work well. no o
positive love     -> Y   | ...this is my second purchase. i used to [LOVE] this toothbrush... my 
money    price    -> Y   | ...re so cute! they dont seem like they're incredibly sturdy, but hone
sadness  dark     -> N   | ... this color and it is better at lasting all day. i love the color, 
negative broken   -> N   | ... broke 30+ yrs ago...pain was gone in minutes ! also rubbed on my s

Model actually used: gpt-5.4-nano-2026-03-17 | temperature=0: True


In [ ]:
OUT = "ai_labels/word_check_ai.csv"

if os.path.exists(OUT):   
    wc = pd.read_csv(OUT, encoding="utf-8-sig", keep_default_na=False)
else:                     
    wc = pd.read_csv("ai_labels/word_check.csv", encoding="utf-8-sig", keep_default_na=False)
    wc = wc[["list", "rank", "word", "sentence", "Y_means", "snippet"]].copy()
    wc["ai_label"], wc["model_used"] = "", ""

todo = wc.index[wc["ai_label"] == ""]
print("Sentences still to label:", len(todo))

for n, i in enumerate(tqdm(todo)):
    label, used = ai_label(wc.at[i, "list"], wc.at[i, "snippet"])
    wc.at[i, "ai_label"], wc.at[i, "model_used"] = label, used
    if n % 50 == 0:
        wc.to_csv(OUT, index=False, encoding="utf-8-sig")   
wc.to_csv(OUT, index=False, encoding="utf-8-sig")

print("\nLabels:", wc["ai_label"].value_counts().to_dict())
print("Not Y/N (should be 0):", (~wc["ai_label"].isin(["Y", "N"])).sum())
print("Model(s) used:", wc["model_used"].unique(), "| date:", datetime.date.today())

Sentences still to label: 0


0it [00:00, ?it/s]


Labels: {'N': 1247, 'Y': 753}
Not Y/N (should be 0): 0
Model(s) used: ['gpt-5.4-nano-2026-03-17'] | date: 2026-10-20


In [ ]:
import os
import numpy as np
import pandas as pd
LIST_ORDER = ["anger", "sadness", "negative", "positive", "money"]

wc = pd.read_csv("ai_labels/word_check_ai.csv", encoding="utf-8-sig", keep_default_na=False)
wc["is_y"] = (wc["ai_label"] == "Y").astype(int)           
wc["is_yn"] = wc["ai_label"].isin(["Y", "N"]).astype(int)   
words = (wc.groupby(["list", "rank", "word"], as_index=False)
           .agg(n_yes=("is_y", "sum"), n=("is_yn", "sum"))) 
words["decision"] = np.where(words["n_yes"] >= words["n"] / 2, "keep", "remove")  

words["checked"] = False
for cat in LIST_ORDER:
    w = words[words["list"] == cat].sort_values("rank")
    stop, in_a_row = None, 0
    for rank, d in zip(w["rank"], w["decision"]):  
        if rank <= 10:                             
            continue
        in_a_row = in_a_row + 1 if d == "keep" else 0
        if in_a_row == 3:
            stop = rank
            break
    last = stop if stop else w["rank"].max()       
    words.loc[(words["list"] == cat) & (words["rank"] <= last), "checked"] = True
    done = words[(words["list"] == cat) & words["checked"]]
    print(f"{cat.upper()}: checked {len(done)} words "
          f"({'stopping rule met' if stop else 'limit of 40 reached'}); "
          f"removed: {done.loc[done['decision'] == 'remove', 'word'].tolist()}")
words.to_csv("ai_labels/word_decisions.csv", index=False)
print("\nSaved word_decisions.csv")

if os.path.exists("manual_checks/blind_check_ai.csv"):
    print("blind_check_ai.csv already exists - not overwritten.")
else:
    checked = set(zip(words.loc[words["checked"], "list"], words.loc[words["checked"], "word"]))
    pool = wc[[(l, w) in checked for l, w in zip(wc["list"], wc["word"])]]
    pool.sample(n=100, random_state=2026)[["list", "word", "Y_means", "snippet"]] \
                .assign(my_mark="").to_csv("manual_checks/blind_check_ai.csv", index=True,
                                   index_label="row_id", encoding="utf-8-sig")
    print("Saved blind_check_ai.csv - type Y or N in the my_mark column")

ANGER: checked 40 words (limit of 40 reached); removed: ['smell', 'bad', 'disappointed', 'feeling', 'remove', 'hot', 'battery', 'broken', 'fits', 'lash', 'honest', 'hurt', 'irritation', 'damage', 'crazy', 'cutting', 'painful', 'hit', 'powerful', 'lose', 'cracked', 'rating', 'tree', 'overpowering', 'wasted', 'stone', 'lace', 'grab', 'shot', 'irritating']
SADNESS: checked 40 words (limit of 40 reached); removed: ['bad', 'leave', 'problem', 'feeling', 'case', 'remove', 'dark', 'broke', 'pain', 'broken', 'bottom', 'fall', 'weight', 'blue', 'fell', 'honest', 'hurt', 'hate', 'irritation', 'damage', 'crazy', 'falling', 'cutting', 'lovely', 'worse', 'awful', 'worn', 'missing', 'negative', 'painful', 'worry', 'lower', 'impossible', 'worried', 'rating']
NEGATIVE: checked 16 words (stopping rule met); removed: ['smell', 'small', 'wear', 'leave', 'feeling', 'case', 'remove', 'battery']
POSITIVE: checked 34 words (stopping rule met); removed: ['found', 'polish', 'top', 'received', 'job', 'full', 'd

In [ ]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd

wc = pd.read_csv("ai_labels/word_check_ai.csv", encoding="utf-8-sig", keep_default_na=False)
mine = pd.read_csv("manual_checks/blind_check_ai.csv", encoding="utf-8-sig", keep_default_na=False)
mine.columns = mine.columns.str.strip()
if "my_mark" not in mine.columns:               
    mine = mine.rename(columns={mine.columns[-1]: "my_mark"})
mine = mine.set_index("row_id")

you = mine["my_mark"].str.strip().str.upper()
ai = wc.loc[mine.index, "ai_label"]
done = you.isin(["Y", "N"])
print("Rows you marked:", done.sum(), "of", len(you))
print("Agreement:", round((ai[done] == you[done]).mean() * 100, 1), "%")
print("Cohen's kappa:", round(cohen_kappa_score(ai[done], you[done]), 2),
      " (0.61+ = substantial; Landis & Koch, 1977)")
diff = mine[done & (ai != you)].assign(ai_label=ai)
print("\nDisagreements:", len(diff))
print(diff[["list", "word", "my_mark", "ai_label", "snippet"]].to_string())

Rows you marked: 100 of 100
Agreement: 84.0 %
Cohen's kappa: 0.63  (0.61+ = substantial; Landis & Koch, 1977)

Disagreements: 16
            list        word my_mark ai_label                                                                                                                                                                             snippet
row_id                                                                                                                                                                                                                           
169        anger  irritation       Y        N  ...rchased! usually within a week my scalp is itching or i have small bumps due to [IRRITATION]. but this hair has allowed me to wear braids for a month or a little longer, in...
121        anger    terrible       N        Y                                                                        ...i have such [TERRIBLE] skin, and at age 55 i need help of course! but i g